In [2]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.transform.mappings import HF_PROVINCE_MAP
from src.transform.normalize import norm_town

print(len(HF_PROVINCE_MAP), "province mappings")
print(norm_town("Bacungan (Leon T. Postigo)"), "|", norm_town("Biñan City"))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
6 province mappings
BACUNGAN | BINAN


In [3]:
RAW = PROJECT_ROOT / "data" / "raw"
memberships = pd.read_parquet(RAW / "hf_memberships.parquet")
winners = pd.read_csv(RAW / "openhalalan_winners.csv")
print(memberships.shape, winners.shape)

(86234, 8) (157333, 14)


In [4]:
memberships["province_std"] = memberships["province"].replace(HF_PROVINCE_MAP)

hf_prov = set(memberships["province_std"].unique())
oh_prov = set(winners["Province"].dropna().unique())
print("Still only in HF:", sorted(hf_prov - oh_prov))

Still only in HF: []


In [5]:
memberships["locality_std"] = memberships["locality"].map(norm_town, na_action="ignore")
winners["town_std"] = winners["City"].map(norm_town, na_action="ignore")

hf_towns = set(memberships["locality_std"].dropna())
oh_towns = set(winners["town_std"].dropna())
print("Still only in HF:", sorted(hf_towns - oh_towns))

Still only in HF: ['MAGADALENA', 'PATEROS TAGUIG', 'PRES CARLOS P GARCIA', 'SAM NARCISO', 'TAGUIG PATEROS']


In [6]:
leftovers = ["MAGADALENA", "PATEROS TAGUIG", "PRES CARLOS P GARCIA", "SAM NARCISO", "TAGUIG PATEROS"]

memberships[memberships["locality_std"].isin(leftovers)].groupby(["province_std", "locality_std"]).agg(
    rows=("id", "size"),
    years=("year", "unique"),
    positions=("position", "unique"),
)

rows               years  \
province_std        locality_std                                     
BOHOL               PRES CARLOS P GARCIA    30  [2004, 2010, 2007]   
LAGUNA              MAGADALENA               1              [2016]   
NCR FOURTH DISTRICT PATEROS TAGUIG           3  [2010, 2004, 2007]   
                    TAGUIG PATEROS           2        [2013, 2016]   
QUEZON              SAM NARCISO              1              [2016]   

                                                                   positions  
province_std        locality_std                                              
BOHOL               PRES CARLOS P GARCIA      [COUNCILOR, MAYOR, VICE MAYOR]  
LAGUNA              MAGADALENA                                   [COUNCILOR]  
NCR FOURTH DISTRICT PATEROS TAGUIG        [MEMBER, HOUSE OF REPRESENTATIVES]  
                    TAGUIG PATEROS        [MEMBER, HOUSE OF REPRESENTATIVES]  
QUEZON              SAM NARCISO                                  [COUNCILOR]

In [7]:
keywords = "MAGDALENA|NARCISO|GARCIA|TAGUIG|PATEROS"
winners.loc[winners["town_std"].str.contains(keywords, na=False), ["Province", "town_std"]].drop_duplicates()

,Province,town_std
1657,BATANGAS,PADRE GARCIA
2212,BOHOL,PRES C P GARCIA
2222,BOHOL,GARCIA HERNANDEZ
8225,LAGUNA,MAGDALENA
10712,NCR FOURTH DISTRICT,TAGUIG
10713,NCR FOURTH DISTRICT,PATEROS
13703,QUEZON,SAN NARCISO
15004,SORSOGON,SANTA MAGDALENA
16575,ZAMBALES,SAN NARCISO
19671,BOHOL,PRES CARLOS P GARCIA PITOGO


In [8]:
from src.transform.mappings import TOWN_MAP

memberships["locality_std"] = memberships["locality_std"].replace(TOWN_MAP)
winners["town_std"] = winners["town_std"].replace(TOWN_MAP)

hf_towns = set(memberships["locality_std"].dropna())
oh_towns = set(winners["town_std"].dropna())
print("Still only in HF:", sorted(hf_towns - oh_towns))
print("Only in OpenHalalan:", len(oh_towns - hf_towns))

Still only in HF: ['TAGUIG PATEROS']
Only in OpenHalalan: 117


In [9]:
TOWN_POSITIONS = ["MAYOR", "VICE MAYOR", "COUNCILOR"]
is_town = memberships["position"].isin(TOWN_POSITIONS)
has_town = memberships["locality_std"].notna()

memberships["locality_source"] = None
memberships.loc[has_town, "locality_source"] = "original"
memberships.loc[~is_town & ~has_town, "locality_source"] = "not_applicable"
memberships["locality_source"].value_counts(dropna=False)

locality_source
original          58594
None              22022
not_applicable     5618
Name: count, dtype: int64

In [10]:
known = memberships[is_town & has_town]
stats = known.groupby(["person_id", "province_std"])["locality_std"].agg(n_towns="nunique", town="first")
one_town = stats.loc[stats["n_towns"] == 1, "town"].rename("history_town").reset_index()
print("person-province pairs with exactly one known town:", len(one_town))

person-province pairs with exactly one known town: 36565


In [11]:
memberships = memberships.drop(columns="history_town", errors="ignore")
memberships = memberships.merge(one_town, on=["person_id", "province_std"], how="left")

is_town = memberships["position"].isin(TOWN_POSITIONS)
fill = is_town & memberships["locality_std"].isna() & memberships["history_town"].notna()
memberships.loc[fill, "locality_std"] = memberships.loc[fill, "history_town"]
memberships.loc[fill, "locality_source"] = "history"

memberships["locality_source"].value_counts(dropna=False)

locality_source
original          58594
history           13795
None               8227
not_applicable     5618
Name: count, dtype: int64

In [12]:
orig = memberships[is_town & (memberships["locality_source"] == "original")]
test = orig[orig["year"] == 2010]
train = orig[orig["year"] != 2010]

train_stats = train.groupby(["person_id", "province_std"])["locality_std"].agg(n_towns="nunique", town="first")
train_one = train_stats.loc[train_stats["n_towns"] == 1, "town"].rename("predicted").reset_index()

check = test.merge(train_one, on=["person_id", "province_std"], how="inner")
accuracy = (check["predicted"] == check["locality_std"]).mean()
print("2010 rows history could fill:", len(check), "of", len(test))
print("accuracy:", round(accuracy * 100, 2), "%")

2010 rows history could fill: 11346 of 16380
accuracy: 99.79 %


### Validation of backfill step 1 (own history)
We hid the known 2010 towns, refilled them from the other years using the same rule (same person, same province, exactly one known town), and compared with the real values.
- **Coverage:** 11,346 of 16,380 town-level rows in 2010 (69%) could be filled.
- **Accuracy:** 99.79% of the filled towns matched the real town (about 24 wrong).

The history rule is reliable, so the 13,795 history fills are kept, labeled `locality_source = "history"`.

In [13]:
persons = pd.read_parquet(RAW / "hf_persons.parquet")

gaps = memberships[memberships["locality_source"].isna()].merge(
    persons, left_on="person_id", right_on="id", suffixes=("", "_person")
)
gaps[["year", "position", "province_std", "first_name", "last_name", "name_suffix"]].head(10)

,year,position,province_std,first_name,last_name,name_suffix
0,2013,COUNCILOR,ABRA,James,Stimson,NaN
1,2016,COUNCILOR,CAGAYAN,David,Sto Tomas,Jr.
2,2016,COUNCILOR,BULACAN,Marciano,Sta Maria,Jr.
3,2013,COUNCILOR,CAMARINES SUR,Anita,Sta Rosa,NaN
4,2013,MAYOR,CEBU,Lissa,Streegan,NaN
5,2016,MAYOR,CEBU,Lissa Marie,Streegan,NaN
6,2013,COUNCILOR,BULACAN,Marciano,Sta Maria,NaN
7,2013,COUNCILOR,BULACAN,Teofilo,Sta Ana,NaN
8,2016,COUNCILOR,ABRA,Manuel James,Stimson,NaN
9,2013,COUNCILOR,CAMARINES SUR,Levi,Staana,NaN


In [14]:
row = gaps.iloc[0]
winners.loc[
    (winners["Year"] == row["year"]) & (winners["Province"] == row["province_std"]),
    ["Last Name", "First Name", "Middle Name", "Title", "Position", "town_std"],
].head(15)

,Last Name,First Name,Middle Name,Title,Position,town_std
68968,ABAOAG,ANICETO,MANZANO,NaN,COUNCILOR,DANGLAS
68969,ABAOAG,ALFONSO,MAYOR,NaN,COUNCILOR,LUBA
68970,ABORJE,ARNULFO,AYUNON,NaN,COUNCILOR,LAGANGILANG
68971,ADAME,DANILO,ABELLA,NaN,COUNCILOR,BANGUED
68972,ADRIATICO,CONCESO,ADRIATICO,NaN,COUNCILOR,LA PAZ
68973,AGAID,JENZY,JAMES BAG-AYAN,NaN,COUNCILOR,SALLAPADAN
68974,AGMATA,ELPIDIO,NaN,NaN,COUNCILOR,SAN ISIDRO
68975,AGSALLONG,ARNOLD,LIZALIS,NaN,COUNCILOR,BOLINEY
68976,AGWAKING,MARIANO,DAKIWAS,NaN,COUNCILOR,TUBO
68977,ALBAYALDE,JULIO,BERNALES,NaN,COUNCILOR,DANGLAS


In [15]:
pattern = r"^(?:STIMSON|STREEGAN|STA|STO|SANTA|SANTO)\b"
mask = winners["Last Name"].str.contains(pattern, na=False) & winners["Year"].isin([2013, 2016])
winners.loc[mask, ["Year", "Province", "Last Name", "First Name", "Middle Name", "Title", "Position", "town_std"]].head(20)

,Year,Province,Last Name,First Name,Middle Name,Title,Position,town_std
69160,2013,ABRA,STIMSON,JAMES,SOTELO,NaN,COUNCILOR,BOLINEY
69253,2013,ABRA,STIMSON,QUERICO,SAMONTE,NaN,VICE MAYOR,BOLINEY
74016,2013,CEBU,STREEGAN,LISSA,NaN,NaN,MAYOR,SOGOD
74088,2013,CEBU,STREEGAN,RICHARD,BUTALID,NaN,VICE MAYOR,SOGOD
86322,2016,ABRA,STIMSON,EDMUND,LUBEN,NaN,COUNCILOR,BOLINEY
86323,2016,ABRA,STIMSON,MANUEL,SOTELO,NaN,COUNCILOR,BOLINEY
89277,2016,BULACAN,STA. MARIA,NONONG,GUZMAN,NaN,COUNCILOR,SAN RAFAEL
89359,2016,BULACAN,STA. ANA,ABS,BINALLA,NaN,VICE MAYOR,BOCAUE
89543,2016,CAGAYAN,STO. TOMAS,DAVID,PEREZ,NaN,COUNCILOR,PIAT
90063,2016,CAMARINES SUR,STA ANA,LEVI,SOLA,NaN,COUNCILOR,CALABANGA


In [16]:
winners["Title"].value_counts().head(10)

Title
ATTY.     339
DOC       299
DATU      180
HADJI      64
BAI        64
PASTOR     53
ENGR.      50
DR.        42
MR.        11
CAPT.       9
Name: count, dtype: int64

In [17]:
from src.transform.normalize import name_key, name_tokens

print(name_key("Sto Tomas"), name_key("STO. TOMAS"))
print(name_key("Staana"), name_key("STA. ANA"))
print(name_key("David Jr."), name_key("DAVID"))
print(name_tokens("Manuel James") & name_tokens("MANUEL"))

STOTOMAS STOTOMAS
STAANA STAANA
DAVID DAVID
{'MANUEL'}


In [18]:
oh = winners[winners["Year"].isin([2013, 2016]) & winners["town_std"].notna()].copy()
oh = oh.rename(columns={"Year": "year", "Position": "position", "Province": "province_std"})
oh["last_key"] = oh["Last Name"].map(name_key, na_action="ignore")
oh["first_key"] = oh["First Name"].map(name_key, na_action="ignore")
oh = oh.dropna(subset=["last_key", "first_key"])

gaps = memberships[memberships["locality_source"].isna()].merge(
    persons, left_on="person_id", right_on="id", suffixes=("", "_person")
)
gaps["last_key"] = gaps["last_name"].map(name_key)
gaps["first_key"] = gaps["first_name"].map(name_key)
print("OpenHalalan rows:", len(oh), "| gap rows:", len(gaps))

OpenHalalan rows: 31511 | gap rows: 8227


In [19]:
KEYS = ["year", "position", "province_std", "last_key"]

t1 = gaps.merge(oh[KEYS + ["first_key", "town_std"]], on=KEYS + ["first_key"], how="inner")
t1_towns = t1.groupby("id")["town_std"].agg(n_towns="nunique", town="first")
t1_ok = t1_towns[t1_towns["n_towns"] == 1]

print("gap rows with a tier 1 match:", len(t1_towns))
print("accepted (exactly one town):", len(t1_ok))

gap rows with a tier 1 match: 5578
accepted (exactly one town): 5575


In [20]:
assert memberships["id"].is_unique

fill_t1 = memberships["id"].isin(t1_ok.index) & memberships["locality_source"].isna()
memberships.loc[fill_t1, "locality_std"] = memberships.loc[fill_t1, "id"].map(t1_ok["town"])
memberships.loc[fill_t1, "locality_source"] = "openhalalan_t1"

memberships["locality_source"].value_counts(dropna=False)

locality_source
original          58594
history           13795
not_applicable     5618
openhalalan_t1     5575
None               2652
Name: count, dtype: int64

In [21]:
gaps2 = gaps[~gaps["id"].isin(t1_ok.index)]
cand = gaps2.merge(oh[KEYS + ["First Name", "town_std"]], on=KEYS, how="inner")

cand["shared"] = [
    bool(name_tokens(a) & name_tokens(b))
    for a, b in zip(cand["first_name"], cand["First Name"])
]
t2 = cand[cand["shared"]]
t2_towns = t2.groupby("id")["town_std"].agg(n_towns="nunique", town="first")
t2_ok = t2_towns[t2_towns["n_towns"] == 1]

print("candidate pairs (same last name):", len(cand))
print("gap rows with a tier 2 match:", len(t2_towns))
print("accepted (exactly one town):", len(t2_ok))

candidate pairs (same last name): 2451
gap rows with a tier 2 match: 642
accepted (exactly one town): 639


In [22]:
fill_t2 = memberships["id"].isin(t2_ok.index) & memberships["locality_source"].isna()
memberships.loc[fill_t2, "locality_std"] = memberships.loc[fill_t2, "id"].map(t2_ok["town"])
memberships.loc[fill_t2, "locality_source"] = "openhalalan_t2"

memberships["locality_source"].value_counts(dropna=False)

locality_source
original          58594
history           13795
not_applicable     5618
openhalalan_t1     5575
None               2013
openhalalan_t2      639
Name: count, dtype: int64

In [23]:
oh_val = winners[(winners["Year"] == 2010) & winners["town_std"].notna()].copy()
oh_val = oh_val.rename(columns={"Year": "year", "Position": "position", "Province": "province_std"})
oh_val["last_key"] = oh_val["Last Name"].map(name_key, na_action="ignore")
oh_val["first_key"] = oh_val["First Name"].map(name_key, na_action="ignore")
oh_val = oh_val.dropna(subset=["last_key", "first_key"])

is_town = memberships["position"].isin(TOWN_POSITIONS)
known10 = memberships[is_town & (memberships["locality_source"] == "original") & (memberships["year"] == 2010)].merge(
    persons, left_on="person_id", right_on="id", suffixes=("", "_person")
)
known10["last_key"] = known10["last_name"].map(name_key)
known10["first_key"] = known10["first_name"].map(name_key)
print("OpenHalalan 2010 rows:", len(oh_val), "| HF 2010 test rows:", len(known10))

OpenHalalan 2010 rows: 15452 | HF 2010 test rows: 16380


In [24]:
v1 = known10.merge(oh_val[KEYS + ["first_key", "town_std"]], on=KEYS + ["first_key"], how="inner")
v1_towns = v1.groupby("id").agg(n_towns=("town_std", "nunique"), predicted=("town_std", "first"), actual=("locality_std", "first"))
v1_ok = v1_towns[v1_towns["n_towns"] == 1]

rest10 = known10[~known10["id"].isin(v1_ok.index)]
vc = rest10.merge(oh_val[KEYS + ["First Name", "town_std"]], on=KEYS, how="inner")
vc["shared"] = [bool(name_tokens(a) & name_tokens(b)) for a, b in zip(vc["first_name"], vc["First Name"])]
v2 = vc[vc["shared"]]
v2_towns = v2.groupby("id").agg(n_towns=("town_std", "nunique"), predicted=("town_std", "first"), actual=("locality_std", "first"))
v2_ok = v2_towns[v2_towns["n_towns"] == 1]

for label, ok in [("Tier 1", v1_ok), ("Tier 2", v2_ok)]:
    acc = (ok["predicted"] == ok["actual"]).mean() * 100
    print(f"{label}: {len(ok)} matched, accuracy {acc:.2f}%")

Tier 1: 13865 matched, accuracy 87.25%
Tier 2: 1532 matched, accuracy 79.70%


In [25]:
wrong1 = v1_ok[v1_ok["predicted"] != v1_ok["actual"]]
print("tier 1 mismatches:", len(wrong1))

leftover_names = oh_towns - hf_towns
print("mismatches where OpenHalalan's town is a leftover name:", wrong1["predicted"].isin(leftover_names).sum())

wrong1.value_counts(["actual", "predicted"]).head(20)

tier 1 mismatches: 1768
mismatches where OpenHalalan's town is a leftover name: 675


actual                       predicted                 
DAVAO                        BANSALAN                      17
ZAMBOANGA                    VINCENZO A SAGUN              15
ANTIPOLO                     CARDONA                       15
CAGAYAN DE ORO               BINUANGAN                     14
CEBU                         CATMON                        14
ILIGAN                       BAROY                         13
TUGUEGARAO                   TUAO                          13
MASBATE                      MANDAON                       12
BAGUIO                       ATOK                          12
BAYBAY                       BATO                          12
ESCALANTE                    ENRIQUE B MAGALONA SARAVIA    12
ISLAND GARDEN CITY OF SAMAL  CARMEN                        12
SORSOGON                     SANTA MAGDALENA               12
PALAYAN                      NAMPICUAN                     12
PANABO                       NEW CORELLA                   12
TAGUIG        

In [26]:
pairs = [("CEBU", "CATMON"), ("CEBU", "CEBU"),
         ("BENGUET", "ATOK"), ("BENGUET", "BAGUIO"),
         ("NCR FOURTH DISTRICT", "PATEROS"), ("NCR FOURTH DISTRICT", "TAGUIG")]

oh_town = oh_val[oh_val["position"].isin(TOWN_POSITIONS)]
oh_counts = oh_town.groupby(["province_std", "town_std"]).size().reindex(pairs)

hf_town = memberships[is_town & (memberships["year"] == 2010)]
hf_counts = hf_town.groupby(["province_std", "locality_std"]).size().reindex(pairs)

pd.DataFrame({"openhalalan_2010": oh_counts, "hf_2010": hf_counts})

openhalalan_2010  hf_2010
province_std                                          
CEBU                CATMON               26.0       10
                    CEBU                  NaN       18
BENGUET             ATOK                 24.0       10
                    BAGUIO                NaN       14
NCR FOURTH DISTRICT PATEROS              37.0       14
                    TAGUIG                NaN       18

In [27]:
big = hf_town.groupby(["province_std", "locality_std"]).size()
big = big[big > 10].index

oh_all = winners[winners["Position"].isin(TOWN_POSITIONS) & winners["town_std"].notna()]
for yr in sorted(oh_all["Year"].unique()):
    present = set(oh_all.loc[oh_all["Year"] == yr, ["Province", "town_std"]].itertuples(index=False, name=None))
    missing = sum(pair not in present for pair in big)
    print(yr, "- big towns missing in OpenHalalan:", missing, "of", len(big))

2001 - big towns missing in OpenHalalan: 1 of 134
2004 - big towns missing in OpenHalalan: 7 of 134
2007 - big towns missing in OpenHalalan: 12 of 134
2010 - big towns missing in OpenHalalan: 133 of 134
2013 - big towns missing in OpenHalalan: 133 of 134
2016 - big towns missing in OpenHalalan: 4 of 134
2019 - big towns missing in OpenHalalan: 4 of 134
2022 - big towns missing in OpenHalalan: 1 of 134
2025 - big towns missing in OpenHalalan: 2 of 134


In [28]:
oh13 = oh_all[oh_all["Year"] == 2013]
oh13_counts = oh13.groupby(["Province", "town_std"]).size()
oh13_counts.index.names = ["province_std", "locality_std"]

hf10_counts = hf_town.groupby(["province_std", "locality_std"]).size()
ratio = oh13_counts / hf10_counts.reindex(oh13_counts.index)

doubled = ratio[ratio > 1.5].index
print("doubled municipalities in OpenHalalan 2013:", len(doubled))

doubled municipalities in OpenHalalan 2013: 112


In [29]:
from_oh = memberships["locality_source"].isin(["openhalalan_t1", "openhalalan_t2"])
in_2013 = memberships["year"] == 2013
in_doubled = pd.MultiIndex.from_frame(memberships[["province_std", "locality_std"]]).isin(doubled)

affected = from_oh & in_2013 & in_doubled
print("OpenHalalan fills by year:")
print(memberships.loc[from_oh, "year"].value_counts())
print("2013 fills pointing to a doubled municipality:", affected.sum())

OpenHalalan fills by year:
year
2013    3885
2016    2329
Name: count, dtype: int64
2013 fills pointing to a doubled municipality: 584


In [30]:
oh16 = oh_all[oh_all["Year"] == 2016]
oh16_counts = oh16.groupby(["Province", "town_std"]).size()
oh16_counts.index.names = ["province_std", "locality_std"]

ratio = oh13_counts / oh16_counts.reindex(oh13_counts.index)
doubled = ratio[ratio > 1.5].index
print("doubled municipalities in OpenHalalan 2013 (vs 2016):", len(doubled))

doubled municipalities in OpenHalalan 2013 (vs 2016): 92


In [31]:
from_oh = memberships["locality_source"].isin(["openhalalan_t1", "openhalalan_t2"])
in_2013 = memberships["year"] == 2013
in_doubled = pd.MultiIndex.from_frame(memberships[["province_std", "locality_std"]]).isin(doubled)
affected = from_oh & in_2013 & in_doubled
print("2013 fills to undo:", affected.sum())

memberships.loc[affected, "locality_std"] = None
memberships.loc[affected, "locality_source"] = "shift_revoked"
memberships["locality_source"].value_counts(dropna=False)

2013 fills to undo: 549


locality_source
original          58594
history           13795
not_applicable     5618
openhalalan_t1     5075
None               2013
openhalalan_t2      590
shift_revoked       549
Name: count, dtype: int64

In [32]:
doubled = oh13_counts[oh13_counts > 16].index
print("doubled (more than 16 officials):", len(doubled))

from_oh = memberships["locality_source"].isin(["openhalalan_t1", "openhalalan_t2"])
in_doubled = pd.MultiIndex.from_frame(memberships[["province_std", "locality_std"]]).isin(doubled)
affected = from_oh & in_2013 & in_doubled
print("additional 2013 fills to undo:", affected.sum())

memberships.loc[affected, "locality_std"] = None
memberships.loc[affected, "locality_source"] = "shift_revoked"
memberships["locality_source"].value_counts(dropna=False)

doubled (more than 16 officials): 94
additional 2013 fills to undo: 4


locality_source
original          58594
history           13795
not_applicable     5618
openhalalan_t1     5071
None               2013
openhalalan_t2      590
shift_revoked       553
Name: count, dtype: int64

In [33]:
valid_pairs = pd.MultiIndex.from_frame(
    memberships.loc[memberships["locality_source"] == "original", ["province_std", "locality_std"]]
).unique()

from_oh = memberships["locality_source"].isin(["openhalalan_t1", "openhalalan_t2"])
pair_ok = pd.MultiIndex.from_frame(memberships[["province_std", "locality_std"]]).isin(valid_pairs)
suspect = from_oh & ~pair_ok

print("OpenHalalan fills with a town never seen in that province in HF:", suspect.sum())
memberships.loc[suspect].value_counts(["year", "province_std", "locality_std"]).head(15)

OpenHalalan fills with a town never seen in that province in HF: 301


year  province_std      locality_std         
2013  ABRA              LICUAN BAAY LICUAN       9
2016  DAVAO OCCIDENTAL  DON MARCELINO            9
      BULACAN           BALAGTAS BIGAA           9
      DAVAO OCCIDENTAL  SANTA MARIA              8
      DINAGAT ISLANDS   BASILISA RIZAL           8
      ABRA              LICUAN BAAY LICUAN       8
2013  NEGROS ORIENTAL   AMLAN AYUQUITAN          7
      LANAO DEL SUR     LUMBA BAYABAO MAGUING    7
2016  CAGAYAN           SANTO NINO FAIRE         7
2013  MAGUINDANAO       SHARIFF AGUAK MAGANOY    7
      DINAGAT ISLANDS   LIBJO ALBOR              6
2016  BOHOL             SAGBAYAN BORJA           6
2013  LANAO DEL SUR     BALINDONG WATU           6
      DINAGAT ISLANDS   BASILISA RIZAL           6
2016  DAVAO OCCIDENTAL  MALITA                   6
Name: count, dtype: int64

In [34]:
valid_by_prov = (
    memberships.loc[memberships["locality_source"] == "original"]
    .groupby("province_std")["locality_std"]
    .apply(set)
)

def suggest(province, town):
    candidates = [t for t in valid_by_prov.get(province, set()) if town.startswith(t + " ")]
    return max(candidates, key=len) if candidates else None

pairs = memberships.loc[suspect, ["province_std", "locality_std"]].drop_duplicates()
pairs["suggestion"] = [suggest(p, t) for p, t in zip(pairs["province_std"], pairs["locality_std"])]

print("unique suspicious pairs:", len(pairs), "| with a suggestion:", pairs["suggestion"].notna().sum())
print(pairs.sort_values(["province_std", "locality_std"]).to_string())

unique suspicious pairs: 65 | with a suggestion: 58
              province_std                                  locality_std                 suggestion
5759                  ABRA                            LICUAN BAAY LICUAN                LICUAN BAAY
12080                ALBAY                                 DARAGA LOCSIN                     DARAGA
11920                ALBAY                           SANTO DOMINGO LIBOG              SANTO DOMINGO
21164              ANTIQUE                            TOBIAS FORNIER DAO             TOBIAS FORNIER
2052                APAYAO                               CALANASAN BAYAG                  CALANASAN
1563                 BOHOL                                SAGBAYAN BORJA                   SAGBAYAN
15095              BULACAN                                BALAGTAS BIGAA                   BALAGTAS
12954              CAGAYAN                              SANTO NINO FAIRE                 SANTO NINO
11742      CAMARINES NORTE                      

In [35]:
ok = pairs.dropna(subset=["suggestion"]).drop_duplicates("locality_std").sort_values("locality_std")
for raw, std in zip(ok["locality_std"], ok["suggestion"]):
    print(f'    "{raw}": "{std}",')

    "ALFONSO LISTA POTIA": "ALFONSO LISTA",
    "AMLAN AYUQUITAN": "AMLAN",
    "ASUNCION SAUG": "ASUNCION",
    "BALAGTAS BIGAA": "BALAGTAS",
    "BALINDONG WATU": "BALINDONG",
    "BASILISA RIZAL": "BASILISA",
    "CALANASAN BAYAG": "CALANASAN",
    "DARAGA LOCSIN": "DARAGA",
    "DATU ODIN SINSUAT DINAIG": "DATU ODIN SINSUAT",
    "DELFIN ALBANO MAGSAYSAY": "DELFIN ALBANO",
    "DON VICTORIANO CHIONGBIAN DON MARIANO MARCOS": "DON VICTORIANO CHIONGBIAN",
    "EL NIDO BACUIT": "EL NIDO",
    "GENERAL TINIO PAPAYA": "GENERAL TINIO",
    "GREGORIO DEL PILAR CONCEPCION": "GREGORIO DEL PILAR",
    "HINOBA AN ASIA": "HINOBA AN",
    "JOSE ABAD SANTOS TRINIDAD": "JOSE ABAD SANTOS",
    "JOSE DALMAN PONOT": "JOSE DALMAN",
    "KABUNTALAN TUMBAO": "KABUNTALAN",
    "LAAK SAN VICENTE": "LAAK",
    "LAMBAYONG MARIANO MARCOS": "LAMBAYONG",
    "LAPU LAPU CITY OPON": "LAPU LAPU",
    "LIBJO ALBOR": "LIBJO",
    "LICUAN BAAY LICUAN": "LICUAN BAAY",
    "LUMBA BAYABAO MAGUING": "LUMBA BAYABAO",
   

In [36]:
print("Lanao del Sur towns starting with TAGOLOAN:",
      sorted(t for t in valid_by_prov["LANAO DEL SUR"] if t.startswith("TAGOLOAN")))

bad = memberships["province_std"].eq("PAMPANGA") & memberships["locality_std"].eq("PAMPANGA")
print("fills with town = PAMPANGA:", bad.sum())
print(memberships.loc[bad, ["year", "position", "locality_source"]].value_counts())

Lanao del Sur towns starting with TAGOLOAN: ['TAGOLOAN II']
fills with town = PAMPANGA: 4
year  position    locality_source
2013  COUNCILOR   openhalalan_t1     3
      VICE MAYOR  openhalalan_t1     1
Name: count, dtype: int64


In [37]:
from src.transform.mappings import TOWN_MAP, PROVINCE_TOWN_MAP

memberships["locality_std"] = memberships["locality_std"].replace(TOWN_MAP)
winners["town_std"] = winners["town_std"].replace(TOWN_MAP)

for (prov, town), std in PROVINCE_TOWN_MAP.items():
    hit = memberships["province_std"].eq(prov) & memberships["locality_std"].eq(town)
    memberships.loc[hit, "locality_std"] = std
    hit_oh = winners["Province"].eq(prov) & winners["town_std"].eq(town)
    winners.loc[hit_oh, "town_std"] = std

bad = memberships["province_std"].eq("PAMPANGA") & memberships["locality_std"].eq("PAMPANGA")
memberships.loc[bad, "locality_std"] = None
memberships.loc[bad, "locality_source"] = "shift_revoked"

from_oh = memberships["locality_source"].isin(["openhalalan_t1", "openhalalan_t2"])
pair_ok = pd.MultiIndex.from_frame(memberships[["province_std", "locality_std"]]).isin(valid_pairs)
suspect = from_oh & ~pair_ok
print("remaining suspicious fills:", suspect.sum())
memberships.loc[suspect].value_counts(["province_std", "locality_std"])

remaining suspicious fills: 33


province_std      locality_std    
DAVAO OCCIDENTAL  DON MARCELINO       9
                  SANTA MARIA         8
                  MALITA              6
                  SARANGANI           5
                  JOSE ABAD SANTOS    5
Name: count, dtype: int64

In [40]:
def prep_hf(rows):
    """Attach person names and match keys to HF membership rows."""
    rows = rows.merge(persons, left_on="person_id", right_on="id", suffixes=("", "_person"))
    rows["last_key"] = rows["last_name"].map(name_key)
    rows["first_key"] = rows["first_name"].map(name_key)
    return rows


def prep_oh(year):
    """OpenHalalan rows for one year that have a town, with match keys."""
    rows = winners[(winners["Year"] == year) & winners["town_std"].notna()].copy()
    rows = rows.rename(columns={"Year": "year", "Position": "position", "Province": "province_std"})
    rows["last_key"] = rows["Last Name"].map(name_key, na_action="ignore")
    rows["first_key"] = rows["First Name"].map(name_key, na_action="ignore")
    return rows.dropna(subset=["last_key", "first_key"])


def match_towns(hf_rows, oh_rows):
    """Tier 1 and Tier 2 matching. Returns the matched town and tier for each HF membership id."""
    t1 = hf_rows.merge(oh_rows[KEYS + ["first_key", "town_std"]], on=KEYS + ["first_key"], how="inner")
    t1 = t1.groupby("id").agg(n_towns=("town_std", "nunique"), town=("town_std", "first"))
    t1 = t1[t1["n_towns"] == 1].assign(tier="t1")

    rest = hf_rows[~hf_rows["id"].isin(t1.index)]
    cand = rest.merge(oh_rows[KEYS + ["First Name", "town_std"]], on=KEYS, how="inner")
    shared = [bool(name_tokens(a) & name_tokens(b)) for a, b in zip(cand["first_name"], cand["First Name"])]
    t2 = cand.loc[shared].groupby("id").agg(n_towns=("town_std", "nunique"), town=("town_std", "first"))
    t2 = t2[t2["n_towns"] == 1].assign(tier="t2")

    return pd.concat([t1, t2])[["town", "tier"]]

In [41]:
is_town = memberships["position"].isin(TOWN_POSITIONS)
known16 = prep_hf(memberships[is_town & (memberships["locality_source"] == "original") & (memberships["year"] == 2016)])

res = match_towns(known16, prep_oh(2016))
res = res.join(known16.set_index("id")["locality_std"].rename("actual"))

for tier, grp in res.groupby("tier"):
    acc = (grp["town"] == grp["actual"]).mean() * 100
    print(f"{tier}: {len(grp)} matched, accuracy {acc:.2f}%")

t1: 6188 matched, accuracy 97.96%
t2: 769 matched, accuracy 97.79%


### Finding: town-label shift in OpenHalalan 2010 and 2013
- In OpenHalalan's 2010 and 2013 data, 133 of 134 cities are missing. Each city's officials are labeled with the municipality listed just before it alphabetically (e.g. Baguio's officials appear under Atok, Taguig's under Pateros).
- Pattern consistent with a forward-fill error: city rows inherited the last municipality name above them. Consecutive cities pile onto the same municipality, and a city at the top of a province's list inherits the province header (e.g. town "PAMPANGA" in Pampanga).
- 2016 and other years are not affected.
- **Action taken:** 2013 OpenHalalan fills pointing to an affected town (more than 16 officials in OpenHalalan 2013) or to the invalid "PAMPANGA" town were undone and labeled `locality_source = "shift_revoked"` (557 rows).

### Validation of OpenHalalan matching
Matching tested on HF rows whose town is known:

| Test year | Tier 1 | Tier 2 |
|---|---|---|
| 2010 (OpenHalalan shifted) | 87.25% | 79.70% |
| 2016 (OpenHalalan reliable) | 97.96% (6,188 rows) | 97.79% (769 rows) |

The matching is reliable. The low 2010 accuracy is caused by the OpenHalalan shift.

### Other notes
- 65 OpenHalalan town spellings with an old name attached (e.g. "AMLAN AYUQUITAN") were mapped to standard names in `TOWN_MAP` after review.
- Davao Occidental towns (Don Marcelino, Malita, Santa Maria, Sarangani, Jose Abad Santos) appear only from 2013. The province was created from Davao del Sur, so these fills are valid.